In [1]:
1+1

2

In [4]:
%pwd

'c:\\Users\\AJAY\\Documents\\ML Projects\\TextSummarizer'

In [3]:
import os

os.chdir("..")

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelevaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_name: Path
    metric_file_name: Path

In [9]:
from src.TextSummarizer.constants import *
import unittest

# Compatibility fix for libraries using the removed Python 2/older Python API
if not hasattr(unittest.TestCase, "assertRaisesRegexp"):
    unittest.TestCase.assertRaisesRegexp = unittest.TestCase.assertRaisesRegex

from src.TextSummarizer.utils.common import read_yaml, create_directories

### Configuration update

In [ ]:
class ConfigurationManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation(self) -> ModelevaluationConfig:
        config = self.config.model_evaluation

        create_directories([self.config.root_dir])

        model_evaluation_config = ModelevaluationConfig(
                root_dir = self.root_dir,
                data_path = self.data_path,
                model_path = self.model_path,
                tokenizer_name = self.tokenizer_name,
                metric_file_name = self.metric_file_name
        )

        return model_evaluation_config

### Components

In [11]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk

import torch
import pandas as pd
from tqdm import tqdm

In [ ]:
import evaluate

rouge_metric = evaluate.load('rouge')

class ModelEvaluation:
    def __init__(self, config: ModelevaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(lself,ist_of_elements, batch_size):
        
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self,dataset, metric, model, tokenizer,
                                batch_size=16, device=device,
                                column_text="article",
                                column_summary="highlights"):
        article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):

            inputs = tokenizer(article_batch, max_length=1024,  truncation=True,
                            padding="max_length", return_tensors="pt")

            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                            attention_mask=inputs["attention_mask"].to(device),
                            length_penalty=0.8, num_beams=8, max_length=128)
            ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''

            # Finally, we decode the generated texts,
            # replace the  token, and add the decoded texts with the references to the metric.
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True)
                for s in summaries]

            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]


            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        #  Finally compute and return the ROUGE scores.
        score = metric.compute()
        return score

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt) 
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

        rouge_metric = rouge_metric


        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt['test'][0:10], rouge_metric, model_pegagus, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
            )

        # Directly use the scores without accessing fmeasure or mid
        rouge_dict = {rn: score[rn] for rn in rouge_names}


        df = pd.DataFrame(rouge_dict, index=[f'pegasus'])
        df.to_csv(self.config.metric_file_name,index=False)




IndentationError: expected an indented block after function definition on line 9 (704344546.py, line 10)